# Análise Exploratória (EDA) — EEG Motor Movement/Imagery Dataset

**Disciplina:** Tratamento de Dados / Engenharia de Dados e Machine Learning
**Dataset:** [EEG Motor Movement/Imagery Dataset (eegmmidb) — PhysioNet](https://physionet.org/content/eegmmidb/1.0.0/)
**Ficha técnica completa:** `../docs/DATASHEET.md`

## Objetivo deste notebook

Explorar uma amostra do dataset para entender:

1. a estrutura do sinal (canais, taxa de amostragem, formato dos arquivos);
2. a cobertura e diversidade dos dados (sujeitos, tarefas, classes de evento);
3. a aparência do sinal bruto e seu conteúdo espectral;
4. inconsistências/qualidade dos dados, antes de qualquer modelagem de ML.

> **Nota sobre reprodutibilidade:** a primeira execução baixa os dados diretamente do
> PhysioNet (acesso aberto, sem necessidade de login) usando `mne.datasets.eegbci`.
> É necessária conexão com a internet. Funciona em ambiente local, Google Colab e
> Kaggle Kernels. Os arquivos ficam em cache em `../data/raw` — execuções seguintes
> não baixam novamente.


## 1. Configuração do ambiente

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

sys.path.insert(0, str(Path("..") / "src"))
import data_loading as dl

mne.set_log_level("WARNING")
warnings.filterwarnings("ignore", category=RuntimeWarning)

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["figure.dpi"] = 100

DATA_DIR = Path("..") / "data" / "raw"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("mne:", mne.__version__)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)


## 2. Parâmetros da amostra explorada

Para manter o notebook rápido de executar, esta EDA usa uma **amostra** de sujeitos
(não os 109 completos) cobrindo os quatro tipos de tarefa do dataset. O pipeline é o
mesmo para o dataset completo — basta ampliar `SUBJECTS`.

Os sujeitos com inconsistências conhecidas (ver `docs/DATASHEET.md`, seção
"Limitações conhecidas") são removidos automaticamente por `dl.valid_subject_ids`.


In [ ]:
SUBJECTS_REQUESTED = [1, 2, 3, 4, 5, 88, 106]  # inclui 2 sujeitos problemáticos de propósito, para demonstrar o filtro
SUBJECTS = dl.valid_subject_ids(SUBJECTS_REQUESTED)
print("Sujeitos solicitados:", SUBJECTS_REQUESTED)
print("Sujeitos válidos após filtro:", SUBJECTS)

# Uma corrida de cada tipo de tarefa: baseline, execução (mãos) e imagética (mãos)
RUNS = [1, 3, 4]
print("Runs analisadas:", RUNS, "-", [dl.RUN_TASK_MAP[r][1] for r in RUNS])


## 3. Download e carregamento dos dados

In [ ]:
raws = {}          # (subject, run) -> mne.io.Raw
summary_rows = []   # linhas de metadados para o DataFrame de resumo

for subject in SUBJECTS:
    for run in RUNS:
        try:
            raw = dl.load_raw(subject, [run], DATA_DIR)
        except Exception as exc:
            print(f"[AVISO] Falha ao baixar/carregar S{subject:03d} R{run:02d}: {exc}")
            continue
        raws[(subject, run)] = raw
        summary_rows.append(dl.raw_summary_row(subject, run, raw))

print(f"{len(raws)} registros (sujeito, run) carregados com sucesso.")


## 4. Inspeção geral de um registro

Vamos olhar a estrutura de um único registro (`Raw` do MNE) em detalhe: canais,
frequência de amostragem, duração e anotações de evento.


In [ ]:
example_key = next(iter(raws))
example_raw = raws[example_key]
print(f"Exemplo: sujeito S{example_key[0]:03d}, run R{example_key[1]:02d}\n")
print(example_raw.info)


In [ ]:
events, event_id = mne.events_from_annotations(example_raw)
print("Mapeamento evento -> código:", event_id)
print("Total de eventos no registro:", events.shape[0])


## 5. Tabela-resumo: cobertura e diversidade da amostra

In [ ]:
df_summary = pd.DataFrame(summary_rows)
df_summary


In [ ]:
print("Sujeitos únicos:", df_summary["subject"].nunique())
print("Tipos de tarefa cobertos:", df_summary["task_type"].unique().tolist())
print("Duração média por registro (s):", df_summary["duration_s"].mean().round(1))
df_summary.groupby("task_type")[["duration_s", "n_events_total"]].describe().T


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
df_summary.groupby("task_type").size().plot(kind="bar", ax=ax, color="#3b6ea5")
ax.set_title("Nº de registros carregados por tipo de tarefa")
ax.set_xlabel("Tipo de tarefa")
ax.set_ylabel("Nº de registros (sujeito x run)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
event_cols = [c for c in df_summary.columns if c.startswith("n_T")]
event_totals = df_summary[event_cols].sum().rename(
    {"n_T0": "T0 (repouso)", "n_T1": "T1", "n_T2": "T2"}
)
fig, ax = plt.subplots(figsize=(6, 4))
event_totals.plot(kind="bar", ax=ax, color="#5a9e6f")
ax.set_title("Balanceamento de classes de evento na amostra")
ax.set_ylabel("Nº total de eventos")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print(event_totals)


## 6. Visualização do sinal bruto

Um trecho de sinal multicanal de um registro de imagética motora, para inspeção
visual de amplitude e ruído.


In [ ]:
imagery_key = next((k for k in raws if k[1] == 4), example_key)
raw_imagery = raws[imagery_key]

fig = raw_imagery.plot(
    duration=8, n_channels=12, scalings=dict(eeg=100e-6), show=False, title=None
)
fig.suptitle(f"Sinal bruto — S{imagery_key[0]:03d} R{imagery_key[1]:02d} (imagética motora)")
plt.show()


## 7. Layout dos eletrodos (montagem espacial)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
example_raw.plot_sensors(show_names=True, axes=ax, show=False)
ax.set_title("Montagem 10-05 — 64 canais EEG")
plt.tight_layout()
plt.show()


## 8. Conteúdo espectral (PSD) por classe de tarefa

Comparação da densidade espectral de potência (PSD) nos canais motores centrais
(C3, Cz, C4) entre os eventos `T1` (mão esquerda) e `T2` (mão direita), em um
registro de imagética motora. Esse tipo de contraste é a base de features clássicas
de BCI (ex.: banda mu 8-12 Hz, banda beta 13-30 Hz sobre o córtex motor).


In [ ]:
motor_channels = [ch for ch in ["C3", "Cz", "C4"] if ch in raw_imagery.ch_names]
events_img, event_id_img = mne.events_from_annotations(raw_imagery)

epochs = mne.Epochs(
    raw_imagery, events_img, event_id=event_id_img,
    tmin=0.0, tmax=4.0, baseline=None, preload=True, picks=motor_channels,
)
print(epochs)

fig, ax = plt.subplots(figsize=(8, 4.5))
colors = {"T1": "#c0504d", "T2": "#4472c4"}
for label in ["T1", "T2"]:
    if label not in epochs.event_id:
        continue
    psd = epochs[label].compute_psd(fmin=1, fmax=40, picks=motor_channels)
    mean_psd = psd.get_data().mean(axis=(0, 1))  # média entre trials e canais
    ax.plot(psd.freqs, 10 * np.log10(mean_psd), label=f"{label}", color=colors.get(label))

ax.axvspan(8, 12, color="gray", alpha=0.15, label="banda mu (8-12 Hz)")
ax.set_xlabel("Frequência (Hz)")
ax.set_ylabel("Potência (dB)")
ax.set_title(f"PSD média em C3/Cz/C4 — S{imagery_key[0]:03d} R{imagery_key[1]:02d}")
ax.legend()
plt.tight_layout()
plt.show()


## 9. Checagem de consistência e dados faltantes

In [ ]:
checks = []
for (subject, run), raw in raws.items():
    checks.append({
        "subject": f"S{subject:03d}",
        "run": f"R{run:02d}",
        "sfreq_ok": raw.info["sfreq"] == dl.SFREQ_EXPECTED,
        "n_channels_ok": len(raw.ch_names) == dl.N_CHANNELS_EXPECTED,
        "has_nan": bool(np.isnan(raw.get_data()).any()),
        "n_bad_channels": len(raw.info["bads"]),
    })

df_checks = pd.DataFrame(checks)
df_checks


In [ ]:
n_ok = (df_checks["sfreq_ok"] & df_checks["n_channels_ok"] & ~df_checks["has_nan"]).sum()
print(f"{n_ok} de {len(df_checks)} registros passaram em todas as checagens básicas "
      f"(sfreq=160Hz, 64 canais, sem NaN).")
assert df_checks["has_nan"].sum() == 0, "Foram encontrados valores NaN em algum registro!"


## 10. Exportar resumo para uso posterior (ML / engenharia de dados)

In [ ]:
out_path = Path("..") / "data" / "eda_summary.csv"
df_summary.to_csv(out_path, index=False)
print(f"Resumo salvo em: {out_path.resolve()}")


## 11. Conclusões da EDA

- O dataset segue rigorosamente a estrutura documentada: 64 canais, 160 Hz, eventos
  `T0`/`T1`/`T2` conforme o tipo de corrida.
- A filtragem dos sujeitos conhecidos como problemáticos (`88`, `106`, entre outros)
  funciona como esperado e deve ser aplicada antes de qualquer modelagem.
- As classes de evento (`T1` vs `T2`) aparecem razoavelmente balanceadas dentro de
  cada registro, o que é favorável para classificação supervisionada sem a
  necessidade de técnicas de balanceamento agressivas.
- O contraste espectral entre `T1` e `T2` na banda mu (8-12 Hz) sobre os canais
  motores é visível mesmo nesta amostra pequena, confirmando que o sinal contém
  informação discriminativa relevante para as tarefas de ML descritas na ficha
  técnica (`docs/DATASHEET.md`, seção 6).
- Não foram encontrados valores ausentes (`NaN`) nos registros carregados; a
  qualidade dos dados brutos, quando os sujeitos problemáticos são excluídos,
  é boa.

**Próximos passos sugeridos para a etapa de ML:**
1. Ampliar a amostra para mais sujeitos (ou o dataset completo) usando o mesmo
   pipeline (`src/data_loading.py`).
2. Extrair features (ex.: banda de potência, CSP) por época/trial.
3. Definir um esquema de validação cross-subject (treinar em alguns sujeitos,
   testar em outros) para avaliar generalização, dado o viés de seleção
   documentado na ficha técnica.
